## Advanced Models and Model Tuning

This notebook uses the Week 6 Dataset A features, including derived ratios and school-district information, to compare a tuned Random Forest with HistGradientBoosting, XGBoost, and LightGBM. The target is the sale price (`ClosePrice`).

**Evaluation plan:** keep the training window fixed at 18 months, select settings using the six chronological validation months from October 2025 through March 2026, then score the selected models once on the April 2026 test month. Candidate selection uses mean validation MAE; RMSE and R² are reported as supporting metrics.


In [1]:
# Imports and project configuration
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid

# These packages must be installed in the active notebook environment.
try:
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError(
        "Install the optional model packages in this kernel with: %pip install xgboost lightgbm"
    ) from exc

import idx_scripts.dataset_a as da
import idx_scripts.features as ft
from idx_scripts.modeling import build_model, fit_and_score
from idx_scripts.splits import validation_months

pd.set_option("display.float_format", "{:,.4f}".format)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

TEST_MONTH = "2026-04"
TRAINING_WINDOW = 18
X = TRAINING_WINDOW
EVAL_MONTHS = validation_months(TEST_MONTH, n_months=6)
RESULTS_DIR = Path("../Rohan/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path("../Data_Prep/CRMLSSoldDatasetA.csv")
DISTRICTS_PATH = Path("DistrictAreas2425.shp")  # relative to the notebook's working directory


In [2]:
# Notebook-level compatibility patch: normalize TargetEncoder(cv=KFold(...)) to cv=5.
import sklearn.preprocessing as sklearn_preprocessing
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder as _OriginalTargetEncoder

if not getattr(da, "_target_encoder_compat_installed", False):
    class NotebookTargetEncoder(_OriginalTargetEncoder):
        def __init__(self, categories="auto", target_type="auto", smooth="auto",
                     cv=5, shuffle=True, random_state=None):
            if not isinstance(cv, (int, np.integer)):
                splitter = cv
                n_splits = getattr(splitter, "n_splits", None)
                if n_splits is None:
                    raise TypeError(
                        "TargetEncoder cv must be an integer or a splitter with n_splits."
                    )
                cv = int(n_splits)
                shuffle = getattr(splitter, "shuffle", shuffle)
                random_state = getattr(splitter, "random_state", random_state)
            super().__init__(
                categories=categories,
                target_type=target_type,
                smooth=smooth,
                cv=cv,
                shuffle=shuffle,
                random_state=random_state,
            )

    # Cover both `from sklearn.preprocessing import TargetEncoder` and a module-level import in dataset_a.
    sklearn_preprocessing.TargetEncoder = NotebookTargetEncoder
    if hasattr(da, "TargetEncoder"):
        da.TargetEncoder = NotebookTargetEncoder
    da._target_encoder_compat_installed = True

# Small smoke check: verify the old KFold-style constructor is normalized correctly.
_encoder_check = NotebookTargetEncoder(
    target_type="continuous", cv=KFold(n_splits=5, shuffle=True, random_state=0)
)
assert _encoder_check.cv == 5
assert _encoder_check.shuffle is True
assert _encoder_check.random_state == 0
print("TargetEncoder KFold compatibility patch is active.")


TargetEncoder KFold compatibility patch is active.


### Load and prepare the data

The feature script adds ratios and spatial school-district features. The custom preprocessor must be told about those extra numeric and categorical columns; otherwise the notebook would appear to run while leaving the new features out of the model. Infinities from zero denominators are converted to missing values before imputation.


In [3]:
districts = ft.load_school_districts(DISTRICTS_PATH)
df = da.load_clean(str(DATA_PATH))
df = ft.add_ratios(df)
df = ft.add_school_districts(df, districts)
df = df.replace([np.inf, -np.inf], np.nan)

EXTRA_NUMERIC = ft.RATIO_COLS + ft.DISTRICT_NUMERIC_COLS
EXTRA_CATEGORICAL = ft.DISTRICT_COLS
MODEL_FEATURES = list(dict.fromkeys(
    da.FEATURES + EXTRA_NUMERIC + EXTRA_CATEGORICAL
))
preprocessor = da.DatasetAPreprocessor(
    extra_numeric=EXTRA_NUMERIC,
    extra_categorical=EXTRA_CATEGORICAL,
)

print(f"Dataset shape: {df.shape}")
print(f"Target: {da.TARGET}; model features: {len(MODEL_FEATURES)}")
print(f"Validation months: {[str(m) for m in EVAL_MONTHS]}; test month: {TEST_MONTH}")


Dataset shape: (309019, 26)
Target: ClosePrice; model features: 23
Validation months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']; test month: 2026-04


### Model factories

All estimators use the same custom preprocessing pipeline and raw-dollar target. The untuned Random Forest settings below provide a same-split reference.


In [4]:
def random_forest(**overrides):
    params = {
        "n_estimators": 200,
        "min_samples_leaf": 2,
        "max_features": 1 / 3,
        "n_jobs": -1,
        "random_state": 0,
    }
    params.update(overrides)
    return RandomForestRegressor(**params)

def hist_gradient_boost(**overrides):
    params = {
        "loss": "squared_error", "learning_rate": 0.1, "max_iter": 200,
        "max_leaf_nodes": 31, "min_samples_leaf": 30,
        "l2_regularization": 0.0, "early_stopping": "auto", "random_state": 0,
    }
    params.update(overrides)
    return HistGradientBoostingRegressor(**params)

def xgboost_regressor(**overrides):
    params = {
        "objective": "reg:squarederror", "tree_method": "hist",
        "n_estimators": 500, "max_depth": 6, "learning_rate": 0.05,
        "subsample": 0.8, "colsample_bytree": 0.8, "n_jobs": -1, "random_state": 0,
    }
    params.update(overrides)
    return XGBRegressor(**params)

def lightgbm_regressor(**overrides):
    params = {
        "objective": "regression", "n_estimators": 500, "learning_rate": 0.05,
        "num_leaves": 31, "max_depth": -1, "min_child_samples": 30,
        "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
        "reg_lambda": 0.0, "n_jobs": -1, "random_state": 0, "verbosity": -1,
    }
    params.update(overrides)
    return LGBMRegressor(**params)


### Chronological tuning helper

The helper evaluates every candidate on exactly the same validation months and training window. Each split refits preprocessing and the model on that split's training rows. It accepts fixed settings so Random Forest can be tuned in two stages: first tree complexity, then `max_features`.


In [5]:
def tune_on_chronological_months(model_name, estimator_factory, parameter_grid, fixed_params=None):
    fixed_params = dict(fixed_params or {})
    rows = []
    fitted_candidates = {}
    candidate_params = {}

    for i, grid_params in enumerate(ParameterGrid(parameter_grid), start=1):
        params = {**fixed_params, **grid_params}
        candidate = f"{model_name} {i:02d}"
        candidate_params[candidate] = params
        model = build_model(estimator_factory(**params), preprocessor=preprocessor)
        fitted_candidates[candidate] = model

        for month in EVAL_MONTHS:
            train, validation = da.split(df, month, TRAINING_WINDOW)
            fitted, _, scores = fit_and_score(
                model, train, validation, features=MODEL_FEATURES, target=da.TARGET
            )
            predictions = fitted.predict(validation[MODEL_FEATURES])
            rows.append({
                "model": model_name, "candidate": candidate, "month": str(month),
                "params": repr(params), "n_train": len(train), "n_validation": len(validation),
                "mae": scores["mae"],
                "rmse": mean_squared_error(validation[da.TARGET], predictions) ** 0.5,
                "r2": scores["r2"], "mape": scores["mape"],
            })

    monthly = pd.DataFrame(rows)
    summary = (
        monthly.groupby(["model", "candidate", "params"], as_index=False)
        .agg(validation_mae=("mae", "mean"), validation_rmse=("rmse", "mean"),
             validation_r2=("r2", "mean"), validation_mape=("mape", "mean"),
             validation_r2_sd=("r2", "std"))
        .sort_values(["validation_mae", "validation_r2"], ascending=[True, False])
        .reset_index(drop=True)
    )
    winner = summary.loc[0, "candidate"]
    return fitted_candidates[winner], candidate_params[winner], summary, monthly


## 1. Random Forest tuning

First compare tree depth, split size, and leaf size. Then hold the best complexity settings fixed and compare the fraction of features considered at each split.


In [6]:
rf_complexity_grid = {
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [2, 10],
}
tuned_rf, rf_complexity_params, rf_complexity_summary, rf_complexity_monthly = (
    tune_on_chronological_months(
        "Random Forest complexity", random_forest, rf_complexity_grid
    )
)
rf_complexity_summary


,model,candidate,params,validation_mae,validation_rmse,validation_r2,validation_mape,validation_r2_sd
0,Random Forest complexity,Random Forest complexity 01,"{'max_depth': None, 'min_samples_leaf': 2, 'mi...","153,595.0069","312,053.7653",0.8900,0.1214,0.0040
1,Random Forest complexity,Random Forest complexity 09,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_...","154,651.2353","313,240.0654",0.8891,0.1226,0.0041
2,Random Forest complexity,Random Forest complexity 02,"{'max_depth': None, 'min_samples_leaf': 2, 'mi...","154,768.2681","314,975.9504",0.8879,0.1222,0.0041
3,Random Forest complexity,Random Forest complexity 10,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_...","155,867.6340","315,741.6754",0.8874,0.1234,0.0041
4,Random Forest complexity,Random Forest complexity 03,"{'max_depth': None, 'min_samples_leaf': 10, 'm...","160,124.5881","325,313.5932",0.8804,0.1261,0.0042
5,Random Forest complexity,Random Forest complexity 04,"{'max_depth': None, 'min_samples_leaf': 10, 'm...","160,124.5881","325,313.5932",0.8804,0.1261,0.0042
6,Random Forest complexity,Random Forest complexity 11,"{'max_depth': 20, 'min_samples_leaf': 10, 'min...","160,419.1941","325,485.2750",0.8803,0.1264,0.0041
7,Random Forest complexity,Random Forest complexity 12,"{'max_depth': 20, 'min_samples_leaf': 10, 'min...","160,419.1941","325,485.2750",0.8803,0.1264,0.0041
8,Random Forest complexity,Random Forest complexity 07,"{'max_depth': 10, 'min_samples_leaf': 10, 'min...","182,924.2016","349,165.1687",0.8623,0.1501,0.0038
9,Random Forest complexity,Random Forest complexity 08,"{'max_depth': 10, 'min_samples_leaf': 10, 'min...","182,924.2016","349,165.1687",0.8623,0.1501,0.0038


In [7]:
rf_feature_grid = {"max_features": [1 / 3, 0.7]}
tuned_rf, best_rf_params, rf_summary, rf_monthly = tune_on_chronological_months(
    "Random Forest", random_forest, rf_feature_grid, fixed_params=rf_complexity_params
)
rf_summary


,model,candidate,params,validation_mae,validation_rmse,validation_r2,validation_mape,validation_r2_sd
0,Random Forest,Random Forest 02,"{'max_depth': None, 'min_samples_leaf': 2, 'mi...","152,817.2368","311,140.6705",0.8906,0.1207,0.0043
1,Random Forest,Random Forest 01,"{'max_depth': None, 'min_samples_leaf': 2, 'mi...","153,595.0069","312,053.7653",0.8900,0.1214,0.0040


In [8]:
rf_monthly.pivot(index="candidate", columns="month", values="mae")


month,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03
candidate,,,,,,
Random Forest 01,"152,851.3613","153,869.9269","149,065.0956","153,506.9617","158,017.5590","154,259.1368"
Random Forest 02,"152,266.7895","152,568.2006","148,628.8380","152,726.5493","156,883.0513","153,829.9920"


## 2. HistGradientBoosting

Compare learning rate and maximum leaf count using the same chronological validation plan.


In [9]:
hgb_grid = {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [15, 31]}
tuned_hgb, best_hgb_params, hgb_summary, hgb_monthly = tune_on_chronological_months(
    "HistGradientBoosting", hist_gradient_boost, hgb_grid
)
hgb_summary


,model,candidate,params,validation_mae,validation_rmse,validation_r2,validation_mape,validation_r2_sd
0,HistGradientBoosting,HistGradientBoosting 04,"{'learning_rate': 0.1, 'max_leaf_nodes': 31}","163,602.3585","311,758.3032",0.8902,0.1353,0.0043
1,HistGradientBoosting,HistGradientBoosting 03,"{'learning_rate': 0.1, 'max_leaf_nodes': 15}","171,818.0081","325,025.1632",0.8807,0.1423,0.0039
2,HistGradientBoosting,HistGradientBoosting 02,"{'learning_rate': 0.03, 'max_leaf_nodes': 31}","178,296.3109","337,623.3441",0.8712,0.1491,0.0045
3,HistGradientBoosting,HistGradientBoosting 01,"{'learning_rate': 0.03, 'max_leaf_nodes': 15}","188,147.9530","354,193.1633",0.8583,0.1580,0.0046


## 3. XGBoost

Compare two tree depths and two learning rates. Other settings are held constant for this first search.


In [10]:
xgb_grid = {"max_depth": [4, 8], "learning_rate": [0.03, 0.1]}
tuned_xgb, best_xgb_params, xgb_summary, xgb_monthly = tune_on_chronological_months(
    "XGBoost", xgboost_regressor, xgb_grid
)
xgb_summary


,model,candidate,params,validation_mae,validation_rmse,validation_r2,validation_mape,validation_r2_sd
0,XGBoost,XGBoost 04,"{'learning_rate': 0.1, 'max_depth': 8}","147,537.9455","293,532.4204",0.9027,0.1190,0.0049
1,XGBoost,XGBoost 02,"{'learning_rate': 0.03, 'max_depth': 8}","152,423.9073","298,349.5945",0.8995,0.1239,0.0040
2,XGBoost,XGBoost 03,"{'learning_rate': 0.1, 'max_depth': 4}","164,201.8566","315,295.9914",0.8877,0.1348,0.0047
3,XGBoost,XGBoost 01,"{'learning_rate': 0.03, 'max_depth': 4}","178,078.6553","339,142.7841",0.8701,0.1466,0.0046


## 4. LightGBM

Compare tree leaf count and minimum child size.


In [11]:
lgbm_grid = {"num_leaves": [15, 31], "min_child_samples": [20, 50]}
tuned_lgbm, best_lgbm_params, lgbm_summary, lgbm_monthly = tune_on_chronological_months(
    "LightGBM", lightgbm_regressor, lgbm_grid
)
lgbm_summary


OSError: exception: access violation reading 0x0000000000000000

## Compare validation results

Check both average validation performance and month-to-month stability before scoring the final test month.


In [ ]:
all_validation = pd.concat(
    [rf_monthly, xgb_monthly, lgbm_monthly, hgb_monthly], ignore_index=True
)
all_summaries = pd.concat(
    [rf_summary, xgb_summary, lgbm_summary, hgb_summary], ignore_index=True
).sort_values("validation_mae").reset_index(drop=True)
all_summaries[["model", "candidate", "params", "validation_mae",
               "validation_rmse", "validation_r2", "validation_r2_sd"]]


## Final test-month comparison

The following cell fits each selected model on the 18 months before April 2026 and evaluates it once on April. Do not change model settings based on these test metrics.


In [ ]:
test_train, test_data = da.split(df, TEST_MONTH, TRAINING_WINDOW)
winners = {
    "Random Forest (untuned reference)": build_model(
        random_forest(), preprocessor=preprocessor
    ),
    "Random Forest (tuned)": tuned_rf,
    "HistGradientBoosting (tuned)": tuned_hgb,
    "XGBoost (tuned)": tuned_xgb,
    "LightGBM (tuned)": tuned_lgbm,
}
validation_winners = {
    "Random Forest (tuned)": rf_summary.iloc[0],
    "HistGradientBoosting (tuned)": hgb_summary.iloc[0],
    "XGBoost (tuned)": xgb_summary.iloc[0],
    "LightGBM (tuned)": lgbm_summary.iloc[0],
}

final_rows, fitted_winners = [], {}
for name, estimator in winners.items():
    start = time.perf_counter()
    fitted, _, test_scores = fit_and_score(
        estimator, test_train, test_data, features=MODEL_FEATURES, target=da.TARGET
    )
    elapsed = time.perf_counter() - start
    fitted_winners[name] = fitted
    predictions = fitted.predict(test_data[MODEL_FEATURES])
    row = {
        "model": name, "training_window_months": TRAINING_WINDOW,
        "test_mae": test_scores["mae"],
        "test_rmse": mean_squared_error(test_data[da.TARGET], predictions) ** 0.5,
        "test_r2": test_scores["r2"], "test_mape": test_scores["mape"],
        "fit_seconds": round(elapsed, 1),
    }
    if name in validation_winners:
        validation = validation_winners[name]
        row.update({
            "validation_mae": validation["validation_mae"],
            "validation_rmse": validation["validation_rmse"],
            "validation_r2": validation["validation_r2"],
            "selected_params": validation["params"],
        })
    else:
        row["selected_params"] = "baseline settings"
    final_rows.append(row)

comparison = pd.DataFrame(final_rows).sort_values("test_mae").reset_index(drop=True)
comparison


## Save results

Write candidate-level validation metrics, the two-stage forest search, and the final test comparison.


In [ ]:
rf_complexity_monthly.to_csv(RESULTS_DIR / "random_forest_complexity_backtest.csv", index=False)
rf_monthly.to_csv(RESULTS_DIR / "random_forest_feature_backtest.csv", index=False)
hgb_monthly.to_csv(RESULTS_DIR / "hist_gradient_boosting_backtest.csv", index=False)
xgb_monthly.to_csv(RESULTS_DIR / "xgboost_backtest.csv", index=False)
lgbm_monthly.to_csv(RESULTS_DIR / "lightgbm_backtest.csv", index=False)
all_summaries.to_csv(RESULTS_DIR / "advanced_model_validation_summary.csv", index=False)
comparison.to_csv(RESULTS_DIR / "advanced_model_test_comparison.csv", index=False)
comparison[["model", "validation_mae", "test_mae", "test_rmse", "test_r2", "fit_seconds"]]
